In [2]:
import numpy as np
from nltk.corpus import gutenberg

text = gutenberg.raw('shakespeare-macbeth.txt')  
corpus = text.split()

vocab = list(set(corpus))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for i,w in enumerate(vocab)}

d = 10  # embedding dimension, your choice
E = np.random.randn(len(vocab), d)  # fresh, random embedding matrix

# fixed-window 

In [3]:
# et: Concatenated embeddings
# getting the embeddings for my words to feed into my nueral net
def build_window_input(words, w2i, E):
    indices = [w2i[w] for w in words]
    vectors = E[indices]
    e_t =vectors.flatten()
    return e_t

words = corpus[:5]
window = build_window_input(words, w2i, E)
print(window.shape)

# ht: hidden representation 
# building ony forward pass of my nn for rnn
np.random.seed(42)
# i want 16 neurons in my hidden layer
W = np.random.randn(16, len(window))
b_h = np.random.randn(16)

def hidden_layer(e_t, W, b_h):
    return np.tanh(W @ e_t + b_h)

# y^_hat: Next-token distribution 
from scipy.special import softmax
U = np.random.randn(len(vocab), 16)   # (|V|, hidden_size)
b_o = np.random.randn(len(vocab))     # (|V|,)


def output_layer(h_t, U, b_o):
    return softmax(U @ h_t +b_o)

(50,)


In [4]:

# 1 FORWARD PASS
# Pass concatenated embeddings through hidden layer
h_t = hidden_layer(window, W, b_h)

# Pass hidden representation through output layer
y_hat = output_layer(h_t, U, b_o)

# checking the model shape
print("Input words:", words)

print("\nShapes:")
print("window e_t:", window.shape)
print("hidden h_t:", h_t.shape)
print("output y_hat:", y_hat.shape)

print("\nProbability check:")
print("Sum of probabilities:", y_hat.sum())

# look at top 5 probs after one forward pass
top_5_indices = np.argsort(y_hat)[-5:][::-1]
print("\nTop 5 predicted next words:")

for idx in top_5_indices:
    print(i2w[idx], y_hat[idx])

Input words: ['[The', 'Tragedie', 'of', 'Macbeth', 'by']

Shapes:
window e_t: (50,)
hidden h_t: (16,)
output y_hat: (5400,)

Probability check:
Sum of probabilities: 1.0

Top 5 predicted next words:
Mortals 0.2654972544655378
on's 0.14914723406352492
shape 0.09833067601300916
Desire 0.0493642168675038
already 0.047881752001799366


# Vanilla RNN

In [32]:
np.random.seed(42)
# embedding dimension
d = 10

# hidden state size: number of neurons / values in the RNN memory
m = 16

# weights for the CURRENT word embedding x_t
W_x = np.random.randn(m, d)

# weights for the PREVIOUS hidden state h_(t-1)
W_h = np.random.randn(m ,m)

# bias for calculating the new hidden state
b_h = np.random.randn(m)

# Weights for my output layer
W_o = np.random.randn(len(w2i), m)

# bias for every output word
b_o = np.random.randn(len(w2i))

def rnn_step(x_t, h_prev, W_x, W_h, b_h):
    h_t = np.tanh(W_h @ h_prev + W_x @ x_t + b_h)
    return h_t

# output layer
def output_layer(h_t, W_o, b_o):
    y_hat = softmax(W_o @ h_t + b_o)
    return y_hat

# calculating the loss 
def nll_loss(y_hat, next_word):
    idx = w2i[next_word]
    next_word_prob = y_hat[idx]
    return -np.log(next_word_prob)

# gradient for softmax output = y hat - yt
def output_gradient(y_hat, next_word):
    one_hot = np.zeros(len(y_hat))
    idx = w2i[next_word]
    one_hot[idx] = 1
    return y_hat - one_hot

# gradient for W_o
def output_weight_gradient(y_hat, next_word, h_t):
    output_grad = output_gradient(y_hat, next_word)
    return np.outer(output_grad, h_t)

# gradient for my ht 
def hidden_output_gradient(W_o, output_grad):
    return np.transpose(W_o.T @ output_grad)

# gradient for tanh
def tanh_gradient(h_t, ht_grad):
    return np.multiply(ht_grad, (1-h_t**2))

# gradient for W_h
def W_h_gradient(h_prev, grad_tanh):
    return np.outer(grad_tanh, h_prev)

# gradient for W_x
def W_x_gradient(x_t, grad_tanh):
    return np.outer(grad_tanh, x_t)

# gradient for h(t-1):
def prev_h_gradient(W_h, grad_tanh):
    return grad_tanh @ W_h

def run_rnn(words):
    # initial hidden state t=0
    h_0 = np.zeros(m)
    h_prev = h_0
    total_loss = 0 
    y_hats = []
    hidden_states = [h_0]
    for t in range(len(words)-1):
        next_word = words[t+1]
        x_t = E[w2i[words[t]]]
        h_t = rnn_step(x_t, h_prev, W_x, W_h, b_h)
        y_hat = output_layer(h_t, W_o, b_o)

        # store FULL probability distribution
        y_hats.append(y_hat)

        # keep track of avg loss
        loss = nll_loss(y_hat, next_word)
        total_loss += loss

        # store hidden state
        hidden_states.append(h_t)

        # recurrent step where my hidden layer at ht uses h_(t-1)
        h_prev = h_t
    avg_loss = total_loss/(len(words)-1)
    # only returns the y_hat for the last word, the in between y_hat gets replaced
    return y_hats, avg_loss, hidden_states


words = corpus[:1000]
y_hats, avg_loss, hidden_states = run_rnn(words)

print(len(y_hats))
print(y_hats[0].shape)
print(len(hidden_states))

999
(5400,)
1000


## One time Back Propogation Through Time

In [ ]:
# initialize accumulators to store gradients across all time steps
grad_W_h = np.zeros((m, m))
grad_W_x = np.zeros((m, d))
grad_W_o = np.zeros((len(w2i), m))

# no future hidden-state error at the final time step
grad_h_future = np.zeros(m)

# get the final prediction, its true target, and the hidden state that produced it
final_y_hat = y_hats[len(words) - 2]
true_next_word = words[len(words) - 1]
h_t = hidden_states[len(words) - 1] # accounted for h0 at the start

# calculate output error: predicted probabilities - true one-hot vector
output_grad = output_gradient(final_y_hat, true_next_word)

# calculate how the final prediction contributes to the gradient of W_o
grad_output_weights = output_weight_gradient(final_y_hat, true_next_word, h_t)

# propagate the output error backwards into the hidden state h_t
grad_h_own = hidden_output_gradient(W_o, output_grad)

# combine this time step's own error with error coming backwards from future states
grad_h_t = grad_h_own + grad_h_future

# propagate the hidden-state error backwards through tanh
grad_tanh = tanh_gradient(h_t, grad_h_t)

x_t_emb = E[w2i[words[3]]]
grad_W_x_next = W_x_gradient(x_t_emb, grad_tanh)

h_prev = hidden_states[len(words) -2]
grad_W_h_next = W_h_gradient(h_prev, grad_tanh)

# pass the error backwards through W_h to the previous hidden state
grad_h_prev = prev_h_gradient(W_h, grad_tanh)

# this becomes the future error when we move to the previous time step
grad_h_future = grad_h_prev

(16, 16)
(16, 10)


## Through the history or words

In [ ]:
# BPTT: calculate and accumulate gradients backwards through all time steps
grad_W_h = np.zeros((m, m))
grad_W_x = np.zeros((m, d))
grad_W_o = np.zeros((len(w2i), m))
grad_h_future = np.zeros(m)
grad_b_o = np.zeros(len(w2i))
grad_b_h = np.zeros(m)

for t in reversed(range(len(words) - 1)):

    # get the values needed for the current time step
    y_hat = y_hats[t]
    next_word = words[t + 1]
    h_t = hidden_states[t + 1]
    h_prev = hidden_states[t]
    x_t = E[w2i[words[t]]]

    # calculate the output error against the true next word
    output_grad = output_gradient(y_hat, next_word)

    # accumulate the gradient for the output weights
    grad_W_o += output_weight_gradient(y_hat, next_word, h_t)

    # propagate the output error backwards into the current hidden state
    grad_h_own = hidden_output_gradient(W_o, output_grad)

    # combine the current prediction error with error coming from future states
    grad_h_t = grad_h_own + grad_h_future

    # propagate the hidden-state error backwards through tanh
    grad_tanh = tanh_gradient(h_t, grad_h_t)

    # accumulate the bias gradients across time
    grad_b_o += output_grad
    grad_b_h += grad_tanh
    
    # accumulate the gradients for the recurrent and input weights
    grad_W_h += W_h_gradient(h_prev, grad_tanh)
    grad_W_x += W_x_gradient(x_t, grad_tanh)

    # pass the hidden-state error backwards to the previous time step
    grad_h_future = prev_h_gradient(W_h, grad_tanh)

## updating the weights, bias, embedding for my vocab through gradient descent

In [33]:
# train the RNN repeatedly by running forward pass, BPTT, and parameter updates
lr = 0.001
epochs = 100

for epoch in range(epochs):

    # forward pass using the CURRENT parameters
    y_hats, avg_loss, hidden_states = run_rnn(words)

    # reset gradients before every backward pass
    grad_W_h = np.zeros_like(W_h)
    grad_W_x = np.zeros_like(W_x)
    grad_W_o = np.zeros_like(W_o)
    grad_b_h = np.zeros_like(b_h)
    grad_b_o = np.zeros_like(b_o)
    grad_E = np.zeros_like(E)

    grad_h_future = np.zeros(m)

    # BPTT through the sequence
    for t in reversed(range(len(words) - 1)):

        y_hat = y_hats[t]
        current_word = words[t]
        next_word = words[t + 1]

        h_t = hidden_states[t + 1]
        h_prev = hidden_states[t]
        x_t = E[w2i[current_word]]

        output_grad = output_gradient(y_hat, next_word)

        grad_W_o += output_weight_gradient(y_hat, next_word, h_t)
        
        grad_b_o += output_grad

        grad_h_own = hidden_output_gradient(W_o, output_grad)

        grad_h_t = grad_h_own + grad_h_future

        grad_tanh = tanh_gradient(h_t, grad_h_t)

        grad_W_h += W_h_gradient(h_prev, grad_tanh)

        grad_W_x += W_x_gradient(x_t, grad_tanh)

        grad_b_h += grad_tanh

        # gradient for the current word's embedding
        grad_x_t = W_x.T @ grad_tanh
        grad_E[w2i[current_word]] += grad_x_t

        # send error backwards through time
        grad_h_future = prev_h_gradient(
            W_h, grad_tanh
        )

    # update all parameters after BPTT
    W_o -= lr * grad_W_o
    W_h -= lr * grad_W_h
    W_x -= lr * grad_W_x

    b_o -= lr * grad_b_o
    b_h -= lr * grad_b_h

    E -= lr * grad_E

    # monitor training
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {avg_loss}")

Epoch 0: Loss = 15.440760211073982
Epoch 10: Loss = 14.35391388671304
Epoch 20: Loss = 13.99795945671804
Epoch 30: Loss = 13.265272653869026
Epoch 40: Loss = 12.962898804711523
Epoch 50: Loss = 12.107839370393053
Epoch 60: Loss = 11.493895319058726
Epoch 70: Loss = 11.020976038052412
Epoch 80: Loss = 10.74308373604891
Epoch 90: Loss = 10.262651139024236


# RNN